

# Prompting Techniques: A Principal-Level Technical Report

## Formal Methods, Mathematical Foundations, and SOTA Algorithmic Formulations for Agentic and RAG-Integrated Prompt Engineering

---

## 1. Formal Foundations and Scope

This report provides a rigorous, production-grade treatment of prompting techniques for Large Language Models (LLMs), formalized through probabilistic frameworks, typed pseudo-algorithms, and architectural integration patterns. Every technique is presented at the state-of-the-art (SOTA) level, with mathematical grounding sufficient for reproduction by researchers and implementation by systems engineers.

**Notation Convention:**

| Symbol | Definition |
|---|---|
| $\mathcal{V}$ | Vocabulary set of the tokenizer |
| $x = (x_1, \ldots, x_n)$ | Input token sequence (prompt) |
| $y = (y_1, \ldots, y_m)$ | Output token sequence (completion) |
| $z = (z_1, \ldots, z_l)$ | Latent reasoning trace (chain of thought) |
| $\theta$ | Model parameters |
| $\mathcal{T}$ | Tool registry |
| $\mathcal{M}$ | Memory state (working, session, episodic, semantic) |
| $\mathcal{R}(\cdot)$ | Retrieval function |
| $B$ | Token budget (hard context-window bound) |
| $\pi$ | Policy (prompt-compiled runtime artifact) |
| $\mathcal{D}_k$ | Demonstration set of $k$ exemplars |

**Core Premise:**

Prompting, at principal-level, is not string concatenation. It is the compilation of a *runtime execution artifact* $\pi$ from structured components—role policy, task objective, protocol bindings, tool affordances, retrieval payloads, memory summaries, and execution state—into a deterministic, token-budgeted prefill that maximizes $P(y^* \mid \pi; \theta)$ for desired output $y^*$.

$$
\pi = \texttt{Compile}\bigl(\text{Role}, \text{Task}, \text{Protocol}, \text{Tools}, \mathcal{R}(q), \mathcal{M}, \text{State}\bigr) \quad \text{s.t.} \quad |\pi| \leq B
$$

---

## 2. Chain-of-Thought (CoT) Prompting: Formal Treatment

### 2.1 Standard Autoregressive Baseline

In a standard autoregressive LLM, the joint probability of a response $y$ conditioned on prompt $x$ decomposes as:

$$
P_\theta(y \mid x) = \prod_{t=1}^{m} P_\theta(y_t \mid y_{<t}, x)
$$

This formulation generates $y$ directly from $x$ without any intermediate reasoning structure, resulting in *opaque single-hop inference*. For compositional, multi-step, or evidence-dense tasks (e.g., multi-document RAG synthesis), this produces systematically degraded accuracy due to the absence of explicit intermediate variable binding.

### 2.2 CoT as Latent Variable Marginalization

Chain-of-Thought prompting introduces a latent reasoning trace $z \in \mathcal{Z}$ such that the model first generates $z$ conditioned on $x$, then generates $y$ conditioned on both $x$ and $z$:

$$
P_\theta(y \mid x) = \sum_{z \in \mathcal{Z}} P_\theta(y \mid z, x) \cdot P_\theta(z \mid x)
$$

In practice, we do not enumerate over $\mathcal{Z}$. We use a *point estimate*: the model samples a single trace $\hat{z} \sim P_\theta(z \mid x)$, and conditions on it:

$$
P_\theta(y \mid x) \approx P_\theta(y \mid \hat{z}, x) \quad \text{where} \quad \hat{z} = \arg\max_z P_\theta(z \mid x)
$$

**Key Insight (Wei et al., 2022; Kojima et al., 2022):** The trace $z$ externalizes intermediate variable bindings that the model would otherwise need to represent implicitly in hidden-state activations. This shifts multi-step composition from implicit attention-head computation to explicit token-space reasoning, where each step is auto-regressively conditioned on prior steps.

### 2.3 SOTA CoT: Compressed, Task-Specific Reasoning Traces

Naive CoT ("Let's think step by step") wastes tokens on verbose natural-language narration. SOTA practice compiles the reasoning specification into a *compressed directive* that constrains the trace structure:

**Definition (Compressed CoT Directive):**

$$
\text{CoT}_{\text{compressed}} = \bigl\langle \text{ReasoningSchema}, \text{MaxSteps}, \text{StepFormat}, \text{TerminationCriterion} \bigr\rangle
$$

**Example Instantiation:**

```
Reason in draft form. Each step: ≤8 words.
Schema: [Identify entities] → [Extract relations] → [Resolve conflicts] → [Synthesize answer]
Max steps: 6. Terminate when answer is unambiguous.
```

This achieves two objectives simultaneously:
1. **Token efficiency:** Output token count for $z$ is bounded by $|z| \leq \text{MaxSteps} \times \text{StepTokenBound}$.
2. **Reasoning faithfulness:** The schema enforces task-relevant cognitive operations rather than unconstrained generation.

### 2.4 Pseudo-Algorithm: CoT-Augmented Generation with Token Budget Enforcement

```
Algorithm 1: CoT-Augmented Generation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x        : user query (token sequence)
  S        : reasoning schema (ordered list of cognitive operations)
  K_max    : maximum reasoning steps
  B_z      : token budget for reasoning trace
  B_y      : token budget for final answer

Output:
  y        : final answer

Procedure:
  1. z ← ∅                                          // Initialize empty trace
  2. FOR i = 1 TO K_max:
       a. s_i ← S[i mod |S|]                        // Select schema operation
       b. z_i ← LLM.generate(
            prompt  = [x; z; "Step {i} ({s_i}):"],
            max_tokens = B_z / K_max,
            stop    = ["\n"]
          )
       c. z ← z ∥ z_i                               // Append step to trace
       d. IF TerminationDetected(z_i):               // Check convergence
            BREAK
       e. IF |z| ≥ B_z:                              // Hard budget enforcement
            BREAK
  3. y ← LLM.generate(
       prompt     = [x; z; "Final Answer:"],
       max_tokens = B_y
     )
  4. RETURN y
```

### 2.5 Mathematical Analysis: When CoT Helps

CoT provides measurable gains when the task requires **compositional generalization**—i.e., when the correct answer $y^*$ is a function of multiple intermediate variables that must be bound sequentially:

$$
y^* = f\bigl(g_1(x), g_2(g_1(x)), \ldots, g_d(\cdots)\bigr)
$$

where $d$ is the **reasoning depth**. Empirically (Feng et al., 2023), standard transformers fail at $d \geq 2$ for out-of-distribution compositions, while CoT maintains accuracy up to the chain length that fits within the context window, because each $g_i$ output is materialized in token space and available for attention at subsequent steps.

**Failure Mode:** CoT degrades when:
- The reasoning schema is misspecified (model follows an incorrect decomposition).
- The trace exceeds the effective attention span, causing early-step information to be lost.
- The model hallucinates intermediate facts that propagate forward.

**Mitigation:** Combine with verification loops (Section 5) and provenance-tagged retrieval (Section 7).

---

## 3. Few-Shot Prompting: Formal Treatment as In-Context Learning

### 3.1 Bayesian Formulation of In-Context Learning

Few-shot prompting provides a demonstration set $\mathcal{D}_k = \{(x_1, y_1), \ldots, (x_k, y_k)\}$ in the context window. The model generates $y$ for a new query $x_{k+1}$ conditioned on both:

$$
P_\theta(y \mid x_{k+1}, \mathcal{D}_k)
$$

**Theoretical Interpretation (Xie et al., 2022; Akyürek et al., 2023):** In-context learning can be understood as *implicit Bayesian inference* over a latent concept variable $c$:

$$
P_\theta(y \mid x_{k+1}, \mathcal{D}_k) = \int P(y \mid x_{k+1}, c) \cdot P(c \mid \mathcal{D}_k) \, dc
$$

The demonstrations $\mathcal{D}_k$ serve as evidence that updates the model's posterior $P(c \mid \mathcal{D}_k)$ over the latent task concept $c$. The model does not update weights; it updates its *effective prior* through attention over the demonstration tokens.

### 3.2 SOTA Exemplar Selection: Optimization over Demonstration Utility

Naive few-shot uses arbitrary or manually curated examples. SOTA practice treats exemplar selection as a combinatorial optimization problem:

**Definition (Exemplar Selection Problem):**

Given a candidate pool $\mathcal{P} = \{(x_i, y_i)\}_{i=1}^{N}$, a query $x_q$, and a quality metric $Q$, select:

$$
\mathcal{D}_k^* = \arg\max_{\mathcal{D}_k \subset \mathcal{P}, |\mathcal{D}_k| = k} Q(x_q, \mathcal{D}_k)
$$

subject to $|\text{Tokenize}(\mathcal{D}_k)| \leq B_{\text{demo}}$ (token budget for demonstrations).

**SOTA Quality Metric (Multi-Signal Ranking):**

$$
Q(x_q, \mathcal{D}_k) = \sum_{i=1}^{k} \Bigl[ \alpha \cdot \text{sim}_{\text{semantic}}(x_q, x_i) + \beta \cdot \text{div}(x_i, \mathcal{D}_k \setminus \{x_i\}) + \gamma \cdot \text{complexity}(y_i) + \delta \cdot \text{recency}(x_i) \Bigr]
$$

where:
- $\text{sim}_{\text{semantic}}$: embedding cosine similarity between the query and exemplar input (ensures relevance).
- $\text{div}$: diversity penalty—maximum marginal relevance (MMR) or DPP-based diversity to avoid redundant demonstrations.
- $\text{complexity}$: reasoning complexity of the exemplar output (prioritize demonstrations that showcase the hardest reasoning the model must replicate).
- $\text{recency}$: temporal freshness for domains with concept drift.

### 3.3 Pseudo-Algorithm: Optimized Few-Shot Construction

```
Algorithm 2: SOTA Few-Shot Exemplar Selection and Prompt Assembly
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x_q        : query
  P          : candidate exemplar pool [(x_i, y_i)]
  k          : number of demonstrations
  B_demo     : token budget for demonstrations
  α, β, γ, δ : ranking weights
  embed(·)   : embedding function

Output:
  D_k        : selected demonstration set
  π_fewshot  : compiled few-shot prompt

Procedure:
  1. // Phase 1: Semantic pre-filter (reduce candidate set)
     e_q ← embed(x_q)
     FOR EACH (x_i, y_i) ∈ P:
       scores[i] ← cosine(e_q, embed(x_i))
     P_filtered ← top-50 by scores[i]       // Pre-filter to top-50

  2. // Phase 2: MMR-based diverse selection
     D_k ← ∅
     FOR j = 1 TO k:
       FOR EACH candidate (x_i, y_i) ∈ P_filtered \ D_k:
         relevance_i ← α · scores[i]
         diversity_i ← β · min_{(x_d,·) ∈ D_k} (1 - cosine(embed(x_i), embed(x_d)))
         complexity_i ← γ · normalized_step_count(y_i)
         recency_i   ← δ · decay(timestamp(x_i))
         combined[i] ← relevance_i + diversity_i + complexity_i + recency_i
       best ← argmax_i combined[i]
       IF |Tokenize(D_k ∪ {best})| > B_demo:
         BREAK                               // Hard token budget enforcement
       D_k ← D_k ∪ {best}

  3. // Phase 3: Order demonstrations by increasing complexity
     D_k ← sort(D_k, key=complexity, order=ascending)

  4. // Phase 4: Compile prompt
     π_fewshot ← Compile(
       role_policy,
       task_objective,
       FORMAT_EACH(D_k as "Input: {x_i}\nOutput: {y_i}"),
       "Input: {x_q}\nOutput:"
     )
  5. RETURN D_k, π_fewshot
```

### 3.4 Demonstration Ordering Effects

Ordering is non-trivial. Lu et al. (2022) demonstrated that permutation of demonstrations can cause accuracy variance of up to 30+ percentage points. SOTA ordering strategies:

| Strategy | Formulation | When to Use |
|---|---|---|
| **Ascending complexity** | Sort $\mathcal{D}_k$ by $\text{complexity}(y_i)$ ascending | General reasoning tasks |
| **Recency-last** | Most recent exemplar placed last (closest to query) | Temporal domains |
| **Similarity-last** | Most similar exemplar placed last | Maximizes recency bias of attention |
| **Curriculum ordering** | Interleave easy and hard examples | Domain adaptation |

---

## 4. Combined CoT + Few-Shot: Formal Synthesis

### 4.1 Mathematical Formulation

The combined technique constructs demonstrations $\mathcal{D}_k$ where each exemplar includes both the input and a *full reasoning trace*:

$$
\mathcal{D}_k^{\text{CoT}} = \bigl\{(x_i, z_i, y_i)\bigr\}_{i=1}^{k}
$$

The model then generates:

$$
P_\theta(z, y \mid x_{k+1}, \mathcal{D}_k^{\text{CoT}}) = P_\theta(z \mid x_{k+1}, \mathcal{D}_k^{\text{CoT}}) \cdot P_\theta(y \mid z, x_{k+1}, \mathcal{D}_k^{\text{CoT}})
$$

This simultaneously conditions on:
1. **Task concept** (from the input-output pairs).
2. **Reasoning structure** (from the trace exemplars).
3. **Output format** (from the final answer format in demonstrations).

### 4.2 Token Budget Trade-Off Analysis

Including traces in demonstrations is token-expensive. The budget constraint becomes:

$$
|\text{Tokenize}(\text{SystemPrompt})| + \sum_{i=1}^{k} \bigl(|x_i| + |z_i| + |y_i|\bigr) + |x_q| + B_{\text{output}} \leq B
$$

**Optimization Strategy:** Compress demonstration traces $z_i$ to their *minimal sufficient* form:

$$
z_i^{\text{compressed}} = \arg\min_{z'} |z'| \quad \text{s.t.} \quad \text{InformationContent}(z', z_i) \geq \tau
$$

In practice, this is achieved by rewriting demonstration traces to use abbreviated notation, dropping filler words, and retaining only the variable bindings and logical transitions.

### 4.3 Pseudo-Algorithm: CoT + Few-Shot Compilation

```
Algorithm 3: CoT + Few-Shot Prompt Compilation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x_q           : query
  D_k_cot       : selected demonstrations with traces [(x_i, z_i, y_i)]
  role_policy   : system role and constraints
  cot_schema    : reasoning schema directive
  B             : total context window budget
  B_output      : reserved output budget

Output:
  π             : compiled prompt

Procedure:
  1. B_available ← B - B_output - |Tokenize(role_policy)| - |Tokenize(cot_schema)| - |x_q|

  2. // Compress traces to fit budget
     FOR i = 1 TO |D_k_cot|:
       z_i_compressed ← CompressTrace(z_i, max_tokens = B_available / (2 * |D_k_cot|))
       demo_i ← Format("Q: {x_i}\nReasoning: {z_i_compressed}\nA: {y_i}")
       IF Σ|demo_j| + |demo_i| > B_available:
         BREAK
       ELSE:
         demos ← demos ∥ demo_i

  3. π ← Concatenate(
       role_policy,
       cot_schema,
       "--- Examples ---",
       demos,
       "--- Now solve ---",
       "Q: {x_q}",
       "Reasoning:"
     )

  4. ASSERT |Tokenize(π)| + B_output ≤ B    // Hard budget invariant
  5. RETURN π
```

---

## 5. Tree of Thoughts (ToT): Formal Search-Theoretic Treatment

### 5.1 Formalization as State-Space Search

Tree of Thoughts (Yao et al., 2023) generalizes CoT from a *single sequential trace* to a *search over a tree of partial reasoning states*. The formalization:

**Definition (ToT State Space):**

$$
\mathcal{G} = (S, A, T, V, s_0)
$$

where:
- $S$: set of **thought states** (partial reasoning traces). Each state $s \in S$ is a sequence of thoughts $s = [z_1, z_2, \ldots, z_j]$.
- $A$: set of **thought generation actions**. Each action $a$ proposes a next thought $z_{j+1}$.
- $T: S \times A \to S$: **transition function** $T(s, a) = s \| z_{j+1}$.
- $V: S \to [0, 1]$: **state value function** (LLM-evaluated heuristic estimating probability that state $s$ leads to correct answer).
- $s_0$: **initial state** (the original query $x$).

### 5.2 Search Algorithms

**Breadth-First Search (BFS-ToT):**

At each depth level $d$, maintain a beam of $b$ states. For each state, generate $k$ candidate thoughts. Evaluate all $b \times k$ candidates with $V(\cdot)$, retain top-$b$.

$$
S_{d+1} = \text{top-}b \Bigl\{ T(s, a) \;\Big|\; s \in S_d, \; a \in \text{Generate}(s, k) \Bigr\} \quad \text{ranked by } V(\cdot)
$$

**Depth-First Search with Backtracking (DFS-ToT):**

Explore one branch fully. If $V(s) < V_{\text{threshold}}$, backtrack to the parent state and try the next candidate thought. This trades breadth for depth and is more token-efficient.

### 5.3 Value Function Design

The value function $V(s)$ is itself an LLM call with a structured evaluation prompt:

$$
V(s) = \text{LLM}\bigl(\text{"Evaluate the following partial reasoning. Rate from 0-1 whether it is on track to solve: "} \| s\bigr)
$$

**SOTA Enhancement: Multi-Criteria Value Function**

$$
V(s) = \omega_1 \cdot \text{Coherence}(s) + \omega_2 \cdot \text{Progress}(s) + \omega_3 \cdot \text{Consistency}(s, \mathcal{R}(x))
$$

where:
- $\text{Coherence}(s)$: logical consistency of the reasoning chain (no contradictions).
- $\text{Progress}(s)$: fraction of sub-questions addressed.
- $\text{Consistency}(s, \mathcal{R}(x))$: agreement with retrieved evidence.

### 5.4 Pseudo-Algorithm: BFS-ToT with Provenance

```
Algorithm 4: BFS Tree-of-Thoughts with Retrieval Consistency
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x           : query
  R           : retrieved evidence set with provenance tags
  b           : beam width
  k           : branching factor (candidates per state)
  D_max       : maximum depth
  V_threshold : minimum value to retain a state
  ω₁, ω₂, ω₃ : value function weights

Output:
  y*          : best answer

Procedure:
  1. S_0 ← {s_0 = (x, R)}                    // Initial state includes query + evidence
  2. FOR d = 0 TO D_max - 1:
       candidates ← ∅
       FOR EACH s ∈ S_d:
         FOR j = 1 TO k:
           // Generate candidate thought conditioned on state + evidence
           z_j ← LLM.generate(
             prompt = [s; "Generate reasoning step {d+1}, candidate {j}:"],
             temperature = 0.7 + 0.1*j     // Increasing diversity across candidates
           )
           s_new ← T(s, z_j)
           candidates ← candidates ∪ {s_new}

       // Evaluate all candidates
       FOR EACH s_c ∈ candidates:
         v_c ← ω₁·Coherence(s_c) + ω₂·Progress(s_c) + ω₃·Consistency(s_c, R)

       // Prune and select beam
       candidates ← {s_c ∈ candidates | v_c ≥ V_threshold}
       S_{d+1} ← top-b(candidates, key=v_c)

       IF any state in S_{d+1} is terminal:     // Terminal = answer is fully derived
         BREAK

  3. s* ← argmax_{s ∈ S_{D_max}} V(s)
  4. y* ← ExtractAnswer(s*)
  5. RETURN y*

Complexity:
  LLM calls per depth level: b·k (generation) + b·k (evaluation) = 2·b·k
  Total LLM calls: O(D_max · b · k)
  Token cost: O(D_max · b · k · avg_state_length)
```

### 5.5 Complexity and Cost Analysis

| Parameter | Typical Value | Impact |
|---|---|---|
| Beam width $b$ | 3–5 | Higher $b$ → better coverage, linear cost increase |
| Branching factor $k$ | 2–5 | Higher $k$ → more diversity, linear cost increase |
| Max depth $D_{\max}$ | 3–6 | Task-dependent; deeper = more expensive |
| Total LLM calls | $2 \cdot b \cdot k \cdot D_{\max}$ | For $b=3, k=3, D_{\max}=4$: **72 calls** |

**Trade-off:** ToT provides substantially higher accuracy on multi-evidence synthesis tasks (10–30% improvement over single-chain CoT on GSM8K, Creative Writing benchmarks from Yao et al., 2023), but at **$O(b \cdot k)$× cost** per depth level. Use ToT selectively for high-stakes, multi-evidence tasks; fall back to single-chain CoT for routine queries.

---

## 6. ReAct Prompting: Formal Agent-Loop Treatment

### 6.1 Formalization as Interleaved Reasoning-Action Traces

ReAct (Yao et al., 2023b) defines agent execution as an alternating sequence of *thoughts* $t_i$, *actions* $a_i$, and *observations* $o_i$:

$$
\tau = (t_1, a_1, o_1, t_2, a_2, o_2, \ldots, t_n, a_n, o_n, y)
$$

where:
- $t_i \in \mathcal{L}$: a natural-language reasoning step (produced by the LLM).
- $a_i \in \mathcal{A}$: an action from the action space (tool call, retrieval query, API invocation).
- $o_i \in \mathcal{O}$: an observation (result returned by the environment).
- $y$: the final synthesized answer.

The probability factorizes as:

$$
P_\theta(\tau \mid x) = \prod_{i=1}^{n} P_\theta(t_i \mid x, \tau_{<i}) \cdot P_\theta(a_i \mid x, \tau_{<i}, t_i) \cdot \underbrace{P_{\text{env}}(o_i \mid a_i)}_{\text{deterministic}} \cdot P_\theta(y \mid x, \tau)
$$

The critical distinction from CoT: the observations $o_i$ are **not generated by the LLM**. They are returned by the external environment (tool execution, retrieval engine, API). This grounds the reasoning chain in real-world state and provides a *self-correction mechanism*—if an action returns unexpected results, the model can reason about the discrepancy and adjust.

### 6.2 SOTA ReAct: Bounded Control Loop with Verification

Production ReAct implementations must enforce the agent loop invariants: **bounded recursion, rollback conditions, failure-state persistence, and verification gates**.

### 6.3 Pseudo-Algorithm: Production-Grade ReAct Loop

```
Algorithm 5: Bounded ReAct Agent Loop with Verification and Failure Recovery
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x           : user query
  T           : tool registry with typed schemas {tool_id → Schema}
  R           : retrieval engine
  N_max       : maximum reasoning-action cycles
  B_ctx       : context window budget
  retry_budget: per-action retry limit
  V_gate      : verification function (returns PASS/FAIL/UNCERTAIN)

Output:
  y           : final answer
  trace       : full execution trace for observability

Procedure:
  1. trace ← []
  2. working_ctx ← CompileInitialContext(x, T.schemas(), R.summary())
  3. FOR i = 1 TO N_max:

       // ── THINK ──
       t_i ← LLM.generate(
         prompt     = [working_ctx; "Thought {i}:"],
         max_tokens = 100,
         stop       = ["Action:"]
       )
       trace.append(("thought", t_i, timestamp()))

       // ── DECIDE ACTION ──
       a_i ← LLM.generate(
         prompt     = [working_ctx; t_i; "Action:"],
         max_tokens = 80,
         stop       = ["Observation:"],
         constrained_grammar = ActionGrammar(T)    // Typed action parsing
       )
       parsed_action ← ParseAction(a_i, T)        // Validate against tool schemas

       IF parsed_action.type == "FINISH":
         y ← parsed_action.answer
         BREAK

       IF parsed_action == PARSE_ERROR:
         trace.append(("error", "Invalid action syntax", timestamp()))
         working_ctx ← AppendRepairDirective(working_ctx, a_i)
         CONTINUE

       // ── EXECUTE ACTION ──
       o_i ← NULL
       FOR attempt = 1 TO retry_budget:
         TRY:
           o_i ← ExecuteTool(
             tool_id    = parsed_action.tool_id,
             params     = parsed_action.params,
             timeout    = T[parsed_action.tool_id].timeout_class,
             auth_scope = CallerScope(x)            // Least privilege
           )
           BREAK
         CATCH ToolError AS e:
           trace.append(("retry", e, attempt, timestamp()))
           IF attempt == retry_budget:
             o_i ← ErrorObservation(e)
           ELSE:
             WAIT(exponential_backoff(attempt) + jitter())

       trace.append(("action", parsed_action, timestamp()))
       trace.append(("observation", o_i, timestamp()))

       // ── VERIFY ──
       v_result ← V_gate(t_i, a_i, o_i, x)
       IF v_result == FAIL:
         trace.append(("verification_fail", v_result.reason, timestamp()))
         working_ctx ← AppendCorrectionDirective(working_ctx, v_result.reason)
         CONTINUE                                  // Re-reason without counting as progress

       // ── CONTEXT MANAGEMENT ──
       working_ctx ← UpdateContext(
         working_ctx,
         new_content = [t_i, a_i, o_i],
         budget      = B_ctx,
         strategy    = "compress_oldest_observations"  // Sliding window with summarization
       )

  4. // ── EXIT ──
     IF i == N_max AND y is undefined:
       y ← "Unable to resolve within {N_max} steps. Partial trace available."
       trace.append(("timeout", N_max, timestamp()))

  5. RETURN y, trace
```

### 6.4 ReAct + RAG Integration: Iterative Retrieval

In RAG contexts, ReAct enables *iterative retrieval refinement*—the model issues a retrieval action, inspects the results, reformulates the query based on what was missing, and retrieves again:

$$
\tau_{\text{RAG}} = (t_1, a_1^{\text{retrieve}}, o_1^{\text{docs}}, t_2^{\text{analyze}}, a_2^{\text{retrieve\_refined}}, o_2^{\text{docs}}, t_3^{\text{synthesize}}, a_3^{\text{FINISH}}, y)
$$

This is strictly superior to single-shot RAG because the model can:
1. Detect insufficient evidence and issue targeted follow-up queries.
2. Detect contradictory evidence and issue disambiguation queries.
3. Detect low-confidence retrievals and fall back to alternative sources.

---

## 7. Prompting for Tool Usage: Typed Contract Formalization

### 7.1 Tool as a Typed Interface

Every tool in a production agentic system is a *first-class typed contract*, not an informal description. The formal specification:

**Definition (Tool Contract):**

$$
\text{Tool} = \Bigl\langle \text{id}, \text{verb}, \text{description}, \Sigma_{\text{in}}, \Sigma_{\text{out}}, \mathcal{C}_{\text{pre}}, \mathcal{C}_{\text{post}}, \text{auth}, \text{timeout\_class}, \text{side\_effects} \Bigr\rangle
$$

| Field | Type | Description |
|---|---|---|
| `id` | `string` | Unique tool identifier (e.g., `get_current_weather`) |
| `verb` | `string` | Active verb summarizing the action |
| `description` | `string` | One-paragraph functional description |
| $\Sigma_{\text{in}}$ | `JSON Schema` | Typed input schema with required/optional fields, formats, constraints |
| $\Sigma_{\text{out}}$ | `JSON Schema` | Typed output schema describing return structure |
| $\mathcal{C}_{\text{pre}}$ | `Predicate[]` | Preconditions that must hold before invocation |
| $\mathcal{C}_{\text{post}}$ | `Predicate[]` | Postconditions guaranteed after successful execution |
| `auth` | `AuthScope` | Required authorization level (caller-scoped, not agent-global) |
| `timeout_class` | `enum{fast,medium,slow}` | Latency tier classification |
| `side_effects` | `enum{none,read,write,delete}` | Mutation classification for approval gating |

### 7.2 SOTA Tool Description Engineering

The LLM's tool selection accuracy is a direct function of the description quality. Formal principles:

**Principle 1: Active Verb Naming**
```
✓  get_current_weather       (clear action)
✗  weather_data              (ambiguous noun)
```

**Principle 2: Schema-Complete Input Specification**
```json
{
  "name": "get_current_weather",
  "description": "Retrieves the current weather conditions for a specified city. Returns temperature (high/low in °C), humidity (%), and conditions (string). Limitation: Only supports cities with population > 100,000. Latency: ~200ms.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name in English (e.g., 'Paris', 'Tokyo')"
      },
      "units": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "default": "celsius"
      }
    },
    "required": ["city"]
  },
  "returns": {
    "type": "object",
    "properties": {
      "high": {"type": "number"},
      "low": {"type": "number"},
      "conditions": {"type": "string"},
      "humidity_pct": {"type": "number"}
    }
  }
}
```

**Principle 3: Explicit Boundary Conditions**

State what the tool *cannot* do. Models exploit negative constraints more reliably than positive-only descriptions:

```
"Limitation: Does not support historical weather queries.
 For historical data, use get_historical_weather instead."
```

**Principle 4: Routing Guidance via Few-Shot**

Include canonical routing examples in the system prompt:

```
Routing Examples:
  "What's the weather in Paris?"        → get_current_weather(city="Paris")
  "Weather last Tuesday in London"      → get_historical_weather(city="London", date="2025-01-07")
  "Restaurant near Eiffel Tower"        → search_places(query="restaurant", near="Eiffel Tower")
  "Translate 'hello' to Japanese"       → DO NOT use weather tools. Use translate_text.
```

### 7.3 Pseudo-Algorithm: Tool Selection and Dispatch

```
Algorithm 6: Typed Tool Selection, Validation, and Dispatch
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  x              : user query
  T              : tool registry [{id, description, Σ_in, Σ_out, C_pre, auth, timeout, side_effects}]
  auth_context   : caller's authorization scope
  approval_gate  : human-in-the-loop approval function (for write/delete tools)

Output:
  result         : tool execution result or error

Procedure:
  1. // ── TOOL SELECTION ──
     available_tools ← FilterByAuth(T, auth_context)
     tool_descriptions ← FormatDescriptions(available_tools)  // Lazy load only relevant schemas

     selection ← LLM.generate(
       prompt = [
         system_policy,
         tool_descriptions,
         routing_examples,
         "Query: {x}",
         "Select tool and parameters as JSON:"
       ],
       response_format = "json_object"
     )

  2. // ── SCHEMA VALIDATION ──
     parsed ← JSON.parse(selection)
     tool ← T[parsed.tool_id]
     IF tool is NULL:
       RETURN Error("Unknown tool: {parsed.tool_id}")

     validation ← JSONSchema.validate(parsed.params, tool.Σ_in)
     IF validation.errors:
       RETURN Error("Parameter validation failed: {validation.errors}")

  3. // ── PRECONDITION CHECK ──
     FOR EACH predicate ∈ tool.C_pre:
       IF NOT predicate.evaluate(parsed.params, auth_context):
         RETURN Error("Precondition failed: {predicate.description}")

  4. // ── APPROVAL GATE (for mutating operations) ──
     IF tool.side_effects ∈ {write, delete}:
       approval ← approval_gate(tool.id, parsed.params, auth_context)
       IF NOT approval.granted:
         RETURN Error("Human approval denied: {approval.reason}")

  5. // ── EXECUTION ──
     result ← ExecuteWithTimeout(
       tool.endpoint,
       parsed.params,
       timeout = tool.timeout_class.duration,
       idempotency_key = hash(x, parsed.tool_id, parsed.params)
     )

  6. // ── OUTPUT VALIDATION ──
     output_valid ← JSONSchema.validate(result, tool.Σ_out)
     IF NOT output_valid:
       RETURN Error("Tool returned invalid output schema")

  7. RETURN result
```

### 7.4 MCP Integration for Tool Discovery

In production agentic architectures, tools are not hardcoded. They are discovered at runtime via the **Model Context Protocol (MCP)**:

```
MCP Discovery Flow:
  1. Agent → MCP Server: ListTools()
     ← Response: [{name, description, inputSchema}]
  
  2. Agent → MCP Server: ListResources()
     ← Response: [{uri, name, mimeType}]
  
  3. Agent → MCP Server: CallTool(name, arguments)
     ← Response: {content: [{type, text}], isError}
```

Tools are loaded lazily—only their names and one-line descriptions are included in the initial context. Full schemas are loaded on-demand when the model signals intent to use a specific tool, minimizing baseline context cost.

---

## 8. Prompt Compilation: The Runtime Artifact Model

### 8.1 Formal Definition

A compiled prompt $\pi$ is a *deterministic, token-budgeted assembly* of structured components:

$$
\pi = \texttt{Compile}\bigl(P_{\text{role}}, P_{\text{task}}, P_{\text{proto}}, P_{\text{tools}}, P_{\text{retrieval}}, P_{\text{memory}}, P_{\text{state}}\bigr)
$$

with the invariant:

$$
\sum_{c \in \{role, task, proto, tools, retrieval, memory, state\}} |P_c| + B_{\text{output}} \leq B
$$

### 8.2 Component Priority Ordering

When the total exceeds $B$, components are trimmed in *reverse priority order*:

| Priority | Component | Trim Strategy |
|---|---|---|
| 1 (highest) | Role Policy $P_{\text{role}}$ | Never trimmed. Contains safety, format, and behavioral constraints. |
| 2 | Task Objective $P_{\text{task}}$ | Never trimmed. Contains the current query and desired outcome. |
| 3 | Protocol Bindings $P_{\text{proto}}$ | Summarize; remove verbose examples. |
| 4 | Tool Affordances $P_{\text{tools}}$ | Lazy-load only relevant tool schemas. |
| 5 | Retrieved Evidence $P_{\text{retrieval}}$ | Rank by provenance score; drop lowest-ranked chunks first. |
| 6 | Memory Summaries $P_{\text{memory}}$ | Compress to key-value summaries; drop episodic details. |
| 7 (lowest) | Execution State $P_{\text{state}}$ | Summarize older steps; retain only last $k$ reasoning steps. |

### 8.3 Pseudo-Algorithm: Prompt Compiler

```
Algorithm 7: Deterministic Prompt Compiler with Token Budget Enforcement
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  components   : ordered list [(label, content, priority, compressor)]
  B            : total token budget
  B_output     : reserved output budget

Output:
  π            : compiled prompt (token sequence)

Procedure:
  1. B_available ← B - B_output
  2. allocated ← {}

  3. // Phase 1: Allocate mandatory components (priority 1-2)
     FOR EACH (label, content, priority, _) WHERE priority ≤ 2:
       tokens ← Tokenize(content)
       allocated[label] ← tokens
       B_available -= |tokens|
       IF B_available < 0:
         RAISE CompilationError("Mandatory components exceed budget")

  4. // Phase 2: Allocate remaining components in priority order
     FOR EACH (label, content, priority, compressor) WHERE priority > 2:
       tokens ← Tokenize(content)
       IF |tokens| ≤ B_available:
         allocated[label] ← tokens
         B_available -= |tokens|
       ELSE:
         // Compress to fit remaining budget
         compressed ← compressor(content, target_tokens=B_available * 0.8)
         IF |Tokenize(compressed)| ≤ B_available:
           allocated[label] ← Tokenize(compressed)
           B_available -= |Tokenize(compressed)|
         ELSE:
           // Drop entirely if cannot fit even after compression
           allocated[label] ← ∅
           LOG.warn("Component {label} dropped due to budget constraints")

  5. // Phase 3: Assemble in canonical order
     π ← Concatenate(
       allocated["role"],
       allocated["task"],
       allocated["protocol"],
       allocated["tools"],
       allocated["retrieval"],
       allocated["memory"],
       allocated["state"]
     )

  6. ASSERT |π| + B_output ≤ B
  7. RETURN π
```

---

## 9. Integration Architecture: Prompting Within the Agentic Stack

### 9.1 End-to-End Execution Flow

The following diagram illustrates how prompting techniques integrate with the full agentic execution stack:

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                        USER REQUEST (JSON-RPC boundary)                     │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  QUERY DECOMPOSITION & REWRITING                                            │
│  • Expand ambiguous queries into explicit sub-queries                       │
│  • Route sub-queries by schema, source, latency tier                        │
│  • Apply CoT decomposition schema for multi-step questions                  │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  HYBRID RETRIEVAL ENGINE                                                    │
│  • Exact match (BM25/keyword)                                              │
│  • Semantic search (dense embedding similarity)                             │
│  • Metadata filters (source authority, freshness, lineage)                  │
│  • Provenance-tagged evidence ranking                                       │
│  • Latency-budgeted execution                                              │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  PROMPT COMPILER (Algorithm 7)                                              │
│  • Assemble: role + task + protocol + tools + evidence + memory + state     │
│  • Enforce token budget B                                                   │
│  • Select prompting technique: CoT / Few-Shot / CoT+Few-Shot / ToT / ReAct │
│  • Compress and prioritize components                                       │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  AGENT EXECUTION LOOP (Algorithm 5: ReAct / Algorithm 4: ToT)              │
│  plan → decompose → retrieve → act → verify → critique → repair → commit   │
│  • Bounded recursion (N_max)                                               │
│  • Verification gates (V_gate)                                             │
│  • Tool dispatch with typed validation (Algorithm 6)                        │
│  • Context window management (sliding window + compression)                 │
│  • Failure state persistence                                                │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  RESPONSE SYNTHESIS & QUALITY GATE                                          │
│  • Final answer extraction                                                  │
│  • Hallucination check against provenance-tagged evidence                   │
│  • Format compliance verification                                           │
│  • Confidence scoring                                                       │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│  MEMORY WRITE-BACK                                                          │
│  • Promote non-obvious corrections to episodic memory                       │
│  • Deduplicate, validate provenance, set expiry                             │
│  • Update session memory with interaction context                           │
└──────────────────────────┬───────────────────────────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                      RESPONSE (JSON-RPC boundary)                           │
│  + Full execution trace for observability                                   │
└──────────────────────────────────────────────────────────────────────────────┘
```

### 9.2 Technique Selection Decision Function

The system must select the appropriate prompting technique at runtime based on query characteristics:

$$
\text{Technique}(x) = \begin{cases}
\text{Direct} & \text{if } \text{complexity}(x) < \tau_1 \text{ and } |\mathcal{R}(x)| \leq 1 \\
\text{CoT} & \text{if } \text{complexity}(x) \geq \tau_1 \text{ and } \text{depth}(x) \leq 3 \\
\text{Few-Shot} & \text{if } \text{domain\_specialization}(x) > \tau_2 \text{ and } \text{complexity}(x) < \tau_1 \\
\text{CoT+Few-Shot} & \text{if } \text{domain\_specialization}(x) > \tau_2 \text{ and } \text{complexity}(x) \geq \tau_1 \\
\text{ToT} & \text{if } \text{ambiguity}(x) > \tau_3 \text{ and } |\mathcal{R}(x)| \geq 3 \\
\text{ReAct} & \text{if } \text{tool\_required}(x) = \text{true} \text{ or } \text{iterative\_retrieval}(x) = \text{true}
\end{cases}
$$

where $\tau_1, \tau_2, \tau_3$ are calibrated thresholds determined by offline evaluation over task-specific benchmarks.

---

## 10. Hallucination Control Through Prompting

### 10.1 Provenance-Grounded Generation

Every retrieved chunk $c_i$ carries a provenance tag $\rho_i = (\text{source}, \text{timestamp}, \text{authority\_score}, \text{chunk\_id})$. The prompt directive enforces citation:

```
INSTRUCTION: Base your answer ONLY on the provided evidence.
For each claim, cite the evidence chunk ID in brackets [chunk_id].
If no evidence supports a claim, state "Insufficient evidence"
rather than generating unsupported content.
```

### 10.2 Self-Consistency Verification

For critical queries, generate $N$ independent CoT traces $\{z^{(1)}, \ldots, z^{(N)}\}$ (by sampling with temperature $T > 0$) and take the majority answer:

$$
y^* = \arg\max_{y} \sum_{j=1}^{N} \mathbb{1}\bigl[\text{ExtractAnswer}(z^{(j)}) = y\bigr]
$$

**Wang et al. (2023):** Self-consistency improves accuracy by 5–18% over single-chain CoT on arithmetic, commonsense, and symbolic reasoning benchmarks, with diminishing returns beyond $N \approx 10$.

### 10.3 Pseudo-Algorithm: Self-Consistency with Cost Cap

```
Algorithm 8: Self-Consistency Decoding with Cost-Bounded Sampling
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  π          : compiled prompt
  N          : maximum sample count
  T          : sampling temperature
  C_max      : maximum token cost
  threshold  : agreement threshold (e.g., 0.7)

Output:
  y*         : majority answer
  confidence : agreement ratio

Procedure:
  1. answers ← []
  2. total_cost ← 0
  3. FOR j = 1 TO N:
       z_j, tokens_used ← LLM.generate(π, temperature=T, return_usage=true)
       total_cost += tokens_used
       y_j ← ExtractAnswer(z_j)
       answers.append(y_j)

       // Early exit if strong consensus already reached
       majority_count ← max(Counter(answers).values())
       IF majority_count / len(answers) ≥ threshold AND len(answers) ≥ 3:
         BREAK

       IF total_cost ≥ C_max:
         BREAK

  4. y* ← mode(answers)
  5. confidence ← count(y* in answers) / len(answers)
  6. RETURN y*, confidence
```

---

## 11. Programmatic Prompt Optimization Frameworks

### 11.1 DSPy: Declarative Prompt Programming

DSPy (Khattab et al., 2023) treats prompting as a *program synthesis* problem rather than manual engineering. Key abstractions:

| DSPy Concept | Formal Analogue | Role |
|---|---|---|
| **Signature** | Type signature $f: \text{Input} \to \text{Output}$ | Declares what the LLM module does |
| **Module** | Parameterized function with learnable prompt parameters | Encapsulates a prompting strategy (CoT, ReAct, etc.) |
| **Teleprompter** | Optimizer $\mathcal{O}$ over prompt parameters | Automatically tunes instructions, few-shot examples, and CoT structure |
| **Metric** | Evaluation function $Q: (y, y^*) \to \mathbb{R}$ | Defines correctness for optimization |

**DSPy Optimization Objective:**

$$
\pi^* = \arg\max_{\pi} \; \mathbb{E}_{(x, y^*) \sim \mathcal{D}_{\text{train}}} \bigl[ Q\bigl(\text{LLM}(x; \pi), y^*\bigr) \bigr]
$$

where $\pi$ includes instructions, demonstrations, and CoT structure—all optimized jointly.

**When to use DSPy:** When you have (a) a measurable quality metric, (b) a training/validation set of 50+ examples, and (c) a pipeline with multiple LLM calls that must be jointly optimized. DSPy eliminates manual prompt iteration and replaces it with automated search.

### 11.2 Comparative Analysis

| Criterion | Manual Prompting | DSPy | LlamaPromptOps | Synalinks |
|---|---|---|---|---|
| **Setup Cost** | Zero | Medium (define signatures, metrics) | Low | Low |
| **Optimization** | Human iteration | Automated (teleprompters) | Template-based | Graph-based |
| **Reproducibility** | Low (ad hoc) | High (deterministic optimization) | Medium | Medium |
| **Multi-stage pipelines** | Manual coordination | Native module composition | Partial | Native |
| **Evaluation integration** | Manual | Built-in metrics and assertions | External | External |
| **Recommended for** | Simple, one-off tasks | Complex multi-stage RAG/agent pipelines | Template standardization | Workflow orchestration |

---

## 12. Summary: Decision Matrix for Prompting Technique Selection

| Technique | Formal Mechanism | Cost Multiplier | Accuracy Gain (vs. Direct) | Best For |
|---|---|---|---|---|
| **Direct** | $P(y \mid x)$ | 1× | Baseline | Simple factual queries, low ambiguity |
| **CoT** | $P(y \mid z, x) \cdot P(z \mid x)$ | 1.3–2× (output tokens) | +10–25% on reasoning tasks | Multi-step reasoning, arithmetic, logic |
| **Compressed CoT** | CoT with $|z| \leq K$ constraint | 1.1–1.5× | +8–22% | Production systems with cost constraints |
| **Few-Shot** | $P(y \mid x, \mathcal{D}_k)$ | 1× (context tokens) | +5–30% on domain tasks | Format compliance, domain specialization |
| **CoT + Few-Shot** | $P(y, z \mid x, \mathcal{D}_k^{\text{CoT}})$ | 1.5–3× (context + output) | +15–35% | Complex domain-specific reasoning |
| **ToT** | BFS/DFS over thought tree | $O(b \cdot k \cdot D_{\max})$ × | +10–30% on multi-path tasks | Ambiguous queries, multi-evidence synthesis |
| **ReAct** | Interleaved $(t, a, o)$ traces | Variable (depends on steps) | Task-dependent | Tool use, iterative retrieval, dynamic tasks |
| **Self-Consistency** | Majority vote over $N$ samples | $N$× | +5–18% | High-stakes tasks requiring reliability |
| **DSPy-Optimized** | Automated $\pi^*$ search | Training cost + 1× inference | +10–40% over manual prompting | Production pipelines with quality metrics |

---

## 13. Production Deployment Considerations

### 13.1 Observability Requirements

Every prompting technique must emit structured traces:

```protobuf
message PromptTrace {
  string   trace_id         = 1;
  string   technique        = 2;   // "cot", "few_shot", "tot", "react"
  int32    total_tokens      = 3;
  int32    prompt_tokens     = 4;
  int32    completion_tokens = 5;
  float    latency_ms        = 6;
  float    confidence        = 7;
  repeated Step steps        = 8;
  string   selected_tool     = 9;
  bool     verification_pass = 10;
  string   failure_reason    = 11;
}
```

### 13.2 Cost Optimization Invariants

1. **Token budget enforcement:** Every compiled prompt $\pi$ must satisfy $|\pi| + B_{\text{output}} \leq B$. Violation is a compilation error, not a runtime surprise.
2. **Technique cost ceiling:** For each technique, define a maximum token cost $C_{\max}^{\text{technique}}$ and enforce it through early termination.
3. **Caching:** Few-shot demonstration blocks and tool schema blocks are *stable across queries*. Cache them as pre-tokenized segments and reuse across compilations.
4. **Prompt-prefix caching:** For models supporting it (e.g., Anthropic prompt caching), structure $\pi$ so that stable components occupy the prefix, minimizing re-processing cost.

### 13.3 Failure Modes and Mitigations

| Failure Mode | Detection | Mitigation |
|---|---|---|
| CoT trace diverges | Step count exceeds $K_{\max}$; value function $V(s) < V_{\min}$ | Hard termination + fallback to direct generation |
| Few-shot exemplar mismatch | Low semantic similarity between selected demos and query | Dynamic re-selection with relaxed diversity constraint |
| ToT combinatorial explosion | Total LLM calls exceed cost ceiling | Reduce beam width $b$ or depth $D_{\max}$ dynamically |
| ReAct infinite loop | Same action repeated $>2$ times | Action deduplication check; force alternative action or FINISH |
| Tool schema validation failure | JSON Schema validation returns errors | Return structured error to LLM for self-repair; max 2 retries |
| Hallucination in final answer | Claim not traceable to any provenance-tagged chunk | Reject answer; re-generate with stricter grounding directive |

---

## 14. Conclusion

Prompting techniques, when formalized as typed, budgeted, and verifiable components of a prompt compilation pipeline, transform from ad hoc string manipulation into rigorous runtime artifacts. The principal-level approach demands:

1. **Mathematical grounding** of each technique's probabilistic mechanism.
2. **Algorithmic specification** with bounded complexity, explicit invariants, and failure recovery.
3. **Integration architecture** that positions prompting within the full agentic stack: retrieval, memory, tool dispatch, verification, and observability.
4. **Production discipline** enforcing token budgets, cost ceilings, schema validation, and structured traces at every boundary.

The techniques—CoT, Few-Shot, ToT, ReAct, and their compositions—are not alternatives to be chosen casually. They are *tools in a typed toolbox*, each with a formal cost-accuracy trade-off profile, to be selected dynamically by a calibrated decision function and executed within a bounded, observable, and recoverable control loop.

---

**References**

- Wei, J. et al. (2022). "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models." *NeurIPS 2022.*
- Kojima, T. et al. (2022). "Large Language Models are Zero-Shot Reasoners." *NeurIPS 2022.*
- Yao, S. et al. (2023). "Tree of Thoughts: Deliberate Problem Solving with Large Language Models." *NeurIPS 2023.*
- Yao, S. et al. (2023b). "ReAct: Synergizing Reasoning and Acting in Language Models." *ICLR 2023.*
- Wang, X. et al. (2023). "Self-Consistency Improves Chain of Thought Reasoning in Language Models." *ICLR 2023.*
- Khattab, O. et al. (2023). "DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines." *arXiv:2310.03714.*
- Xie, S. M. et al. (2022). "An Explanation of In-Context Learning as Implicit Bayesian Inference." *ICLR 2022.*
- Akyürek, E. et al. (2023). "What learning algorithm is in-context learning? Investigations with linear models." *ICLR 2023.*
- Lu, Y. et al. (2022). "Fantastically Ordered Prompts and Where to Find Them." *ACL 2022.*
- Feng, G. et al. (2023). "Towards Revealing the Mystery behind Chain of Thought." *NeurIPS 2023.*



# Prompting Techniques for Agentic AI Systems: A Principal-Level Technical Report

---

## 1. Executive Summary

This report provides a rigorous, state-of-the-art (SOTA) treatment of prompting techniques as they apply to production-grade agentic AI systems, Retrieval-Augmented Generation (RAG) pipelines, and multi-agent orchestration frameworks. Each technique is formalized with mathematical foundations, pseudo-algorithmic specifications, typed interface contracts, and trade-off analyses suitable for principal-level engineers, applied researchers, and system architects operating at enterprise scale.

The distinction between **prompt engineering** and **context engineering** is architecturally critical: prompt engineering governs the _instruction surface_ presented to the language model, while context engineering governs the _information substrate_ — retrieved evidence, memory summaries, tool affordances, execution state — assembled within the bounded token budget. The techniques presented herein achieve maximum efficacy only when both surfaces are co-optimized as a single compiled runtime artifact.

---

## 2. Formal Foundations: Prompting as Conditional Distribution Steering

### 2.1 Mathematical Framework

Let $\mathcal{V}$ denote the vocabulary of a language model $\mathcal{M}$ with parameters $\theta$. An autoregressive LLM defines a conditional distribution:

$$P_\theta(y_t \mid y_{<t}, x) = \text{softmax}\left(\frac{W_o \cdot h_t^{(L)}}{\tau}\right)$$

where $x$ is the input prompt (context window contents), $y_{<t}$ is the sequence of previously generated tokens, $h_t^{(L)}$ is the final-layer hidden state at position $t$, $W_o$ is the output projection matrix, and $\tau$ is the temperature parameter.

A **prompt** $\mathcal{P}$ is a structured prefix $x = \mathcal{P} \oplus \mathcal{C}$, where $\mathcal{P}$ is the instruction/example surface and $\mathcal{C}$ is the context payload (retrieved documents, memory, tool schemas). The objective of prompt engineering is to find:

$$\mathcal{P}^* = \arg\max_{\mathcal{P} \in \mathbb{P}} \; \mathbb{E}_{q \sim \mathcal{Q}} \left[ U\big(y^*_q, \; \mathcal{M}(q, \mathcal{P}, \mathcal{C}_q)\big) \right]$$

subject to:

$$|\text{tokens}(\mathcal{P}) + \text{tokens}(\mathcal{C}_q)| \leq B_{\text{ctx}} - B_{\text{gen}}$$

where:
- $\mathbb{P}$ is the space of valid prompt templates
- $\mathcal{Q}$ is the query distribution
- $U(\cdot, \cdot)$ is a task-specific utility function (e.g., F1, BLEU, faithfulness score, tool-call accuracy)
- $B_{\text{ctx}}$ is the model's context window size
- $B_{\text{gen}}$ is the reserved generation budget

### 2.2 The Token Budget Constraint as a Hard Architectural Invariant

The token budget is not advisory — it is a **hard memory wall**. Every prompting technique must be evaluated against its marginal utility per token consumed:

$$\text{Efficiency}(\mathcal{P}_i) = \frac{\Delta U(\mathcal{P}_i)}{\Delta T(\mathcal{P}_i)}$$

where $\Delta U(\mathcal{P}_i)$ is the utility gain from incorporating prompting technique $i$ and $\Delta T(\mathcal{P}_i)$ is the additional token cost. Techniques that produce high $\Delta U$ with low $\Delta T$ are architecturally preferred. This metric governs all design decisions in this report.

---

## 3. Chain-of-Thought (CoT) Prompting: Formalization and SOTA Practices

### 3.1 Theoretical Basis

Chain-of-Thought prompting exploits the observation that autoregressive models can solve problems requiring multi-step reasoning only when the intermediate computation is **externalized into the token stream**. Without CoT, the model must compress $k$ reasoning steps into a single forward pass from question to answer, which exceeds the representational capacity of a fixed-depth transformer for sufficiently complex tasks.

**Formal Characterization.** Let a reasoning task require a derivation chain $\mathcal{R} = (r_1, r_2, \ldots, r_k)$ to arrive at answer $a$ from question $q$ given evidence $\mathcal{E}$. Standard prompting models $P(a \mid q, \mathcal{E})$ directly. CoT models:

$$P_{\text{CoT}}(a \mid q, \mathcal{E}) = \sum_{\mathcal{R}} P(a \mid \mathcal{R}, q, \mathcal{E}) \cdot P(\mathcal{R} \mid q, \mathcal{E})$$

In practice, greedy decoding selects the single most likely chain $\hat{\mathcal{R}}$, and the answer is conditioned on it:

$$\hat{\mathcal{R}} = \arg\max_{\mathcal{R}} P(\mathcal{R} \mid q, \mathcal{E}), \quad \hat{a} = \arg\max_a P(a \mid \hat{\mathcal{R}}, q, \mathcal{E})$$

This decomposition transforms a hard single-step inference into a sequence of easier conditional inferences, each of which is within the model's per-step capacity.

### 3.2 SOTA CoT Variants

| Variant | Mechanism | When to Use |
|---|---|---|
| **Zero-Shot CoT** | Append "Let's think step by step." | Baseline; low token cost, moderate accuracy gain |
| **Manual CoT** | Hand-crafted reasoning chains in prompt | High-stakes domains requiring domain-specific logic |
| **Auto-CoT** (Zhang et al., 2022) | Cluster questions, generate diverse chains automatically | Scale across heterogeneous query distributions |
| **Complexity-Based CoT** (Fu et al., 2023) | Sample multiple chains, select the one with highest step count among correct candidates | Tasks where longer reasoning correlates with correctness |
| **Contrastive CoT** | Provide both correct and incorrect reasoning chains with explicit labels | Error-prone domains where the model must learn to avoid specific failure modes |
| **Compressed CoT** | Constrain reasoning to abbreviated symbolic notation | Token-budget-constrained production systems |

### 3.3 Compressed CoT: Production-Grade Token Optimization

In production agentic systems, verbose CoT chains consume significant generation budget. **Compressed CoT** constrains the model's reasoning to a draft form, using abbreviated notation, symbolic operators, or keyword-only traces.

**Formal Constraint:**

$$|\text{tokens}(\mathcal{R}_{\text{compressed}})| \leq \alpha \cdot |\text{tokens}(\mathcal{R}_{\text{verbose}})|, \quad \alpha \in [0.15, 0.35]$$

**Implementation Specification:**

```
SYSTEM INSTRUCTION (Compressed CoT Directive):

When reasoning, use DRAFT MODE:
- Maximum 5 words per reasoning step
- Use symbolic operators: ∵ (because), ∴ (therefore), ⊕ (combine), ✗ (reject), ✓ (accept)
- Number each step
- Emit final answer on last line prefixed with "ANSWER:"

Example:
Q: Given Doc A states revenue=$5M (2023) and Doc B states revenue=$4.8M (2023), what is the revenue?
DRAFT:
1. DocA: $5M, source=annual_report ✓
2. DocB: $4.8M, source=press_release
3. ∵ annual_report > press_release authority ∴ prefer DocA
4. Conflict flag: $200K discrepancy noted
ANSWER: $5M (per annual report; $200K discrepancy with press release noted)
```

**Trade-off Analysis:**

| Dimension | Verbose CoT | Compressed CoT |
|---|---|---|
| Token cost (generation) | High (~200-500 tokens) | Low (~40-120 tokens) |
| Reasoning transparency | Full natural language | Requires operator vocabulary |
| Auditability | Direct human reading | Requires trained reviewers |
| Accuracy retention | Baseline | ~95-98% of verbose (task-dependent) |
| Latency (TTFT impact) | Higher | Lower |
| Cost per inference | Higher | ~60-75% reduction on generation side |

### 3.4 Pseudo-Algorithm: CoT-Augmented RAG Inference

```
ALGORITHM: ChainOfThought_RAG_Inference

INPUT:
  q         : UserQuery
  E         : RetrievedEvidenceSet (provenance-tagged)
  P_role    : RolePolicy
  B_ctx     : ContextWindowBudget
  B_gen     : GenerationBudget
  mode      : {VERBOSE, COMPRESSED}

OUTPUT:
  answer    : StructuredAnswer
  chain     : ReasoningTrace
  confidence: float ∈ [0, 1]

PROCEDURE:
  1. ESTIMATE token_cost(P_role) + token_cost(E) + token_cost(CoT_instruction)
  2. IF estimated_total > B_ctx - B_gen:
       COMPRESS E by removing low-relevance documents (below threshold τ_rel)
       TRUNCATE E to fit within budget
  3. CONSTRUCT prefill:
       prefill ← CONCAT(P_role, CoT_instruction[mode], E_with_provenance, q)
  4. GENERATE (chain, answer) ← M.generate(prefill, max_tokens=B_gen, temperature=τ)
  5. PARSE answer from chain using structured output parser
  6. VALIDATE:
       a. Check answer is grounded in E (faithfulness verification)
       b. Check reasoning steps reference specific evidence items by provenance ID
       c. Compute confidence = f(num_supporting_docs, agreement_score, source_authority)
  7. IF confidence < θ_min:
       FLAG for human review or trigger REPAIR loop
  8. RETURN (answer, chain, confidence)
```

---

## 4. Few-Shot Prompting: Formalization and SOTA Practices

### 4.1 Theoretical Basis: In-Context Learning as Implicit Bayesian Inference

Few-shot prompting leverages **in-context learning (ICL)**, where the model implicitly infers a task distribution from $k$ input-output demonstration pairs $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{k}$ without gradient updates.

Recent theoretical analysis (Xie et al., 2022; Akyürek et al., 2023) demonstrates that ICL in transformers approximates Bayesian inference over a latent task concept $\phi$:

$$P(y \mid x, \mathcal{D}) \approx \int P(y \mid x, \phi) \cdot P(\phi \mid \mathcal{D}) \, d\phi$$

The model uses the demonstrations to form a posterior $P(\phi \mid \mathcal{D})$ over possible task specifications, then generates $y$ conditioned on the inferred task. This is **not** few-shot fine-tuning; no weights change. It is an inference-time computation within the context window.

### 4.2 SOTA Few-Shot Selection Strategies

Naive few-shot (random example selection) is suboptimal. SOTA systems use **dynamic, query-conditioned example selection**:

#### 4.2.1 Similarity-Based Selection (KATE — Khandelwal-inspired Adaptive Top-K Examples)

Select $k$ demonstrations whose inputs $x_i$ are most similar to the current query $q$ in embedding space:

$$\mathcal{D}_q = \text{TopK}_{(x_i, y_i) \in \mathcal{D}_{\text{pool}}} \; \text{sim}(\text{emb}(q), \text{emb}(x_i))$$

where $\text{sim}(\cdot, \cdot)$ is cosine similarity over a calibrated embedding model (e.g., `text-embedding-3-large`, `voyage-3-large`).

#### 4.2.2 Diversity-Augmented Selection

Pure similarity can yield redundant examples. Apply **Maximal Marginal Relevance (MMR)** to balance relevance and diversity:

$$\text{MMR}(x_i) = \lambda \cdot \text{sim}(q, x_i) - (1 - \lambda) \cdot \max_{x_j \in \mathcal{D}_{\text{selected}}} \text{sim}(x_i, x_j)$$

where $\lambda \in [0.5, 0.8]$ balances relevance against diversity.

#### 4.2.3 Difficulty-Stratified Selection

Include examples spanning easy, medium, and hard difficulty levels for the target task. This prevents the model from anchoring on trivially simple patterns and improves generalization to edge cases.

#### 4.2.4 Contrastive Examples (Positive + Negative)

Include both correct demonstrations and explicitly labeled incorrect demonstrations:

```
CORRECT EXAMPLE:
  Input: "What is the capital of France?"
  Output: "Paris" ✓
  Reasoning: Direct factual lookup, single unambiguous answer.

INCORRECT EXAMPLE (DO NOT FOLLOW):
  Input: "What is the capital of France?"
  Output: "France is a country in Europe with many cities." ✗
  Error: Failed to provide a direct answer; generated tangential information.
```

This technique reduces specific failure modes by up to 35% in empirical evaluations on domain-specific benchmarks.

### 4.3 Token-Efficient Few-Shot: Structural Optimization

Each few-shot example consumes tokens. In a budget-constrained system, we must minimize per-example token cost while preserving the signal that enables ICL.

**Techniques:**

1. **Minimal Pair Examples:** Strip examples to the minimum tokens that distinguish correct from incorrect behavior.
2. **Tabular Encoding:** Encode examples as structured tables rather than prose.
3. **Dynamic k-Selection:** Vary the number of examples based on available token budget:

$$k^* = \min\left(k_{\max}, \left\lfloor \frac{B_{\text{ctx}} - |\text{tokens}(\mathcal{P}_{\text{fixed}})| - |\text{tokens}(\mathcal{C}_q)| - B_{\text{gen}}}{|\text{tokens}_{\text{avg}}(\text{example})|} \right\rfloor \right)$$

4. **Example Caching with Hash Keys:** Precompute and cache token-encoded examples indexed by query cluster, avoiding runtime tokenization overhead.

### 4.4 Pseudo-Algorithm: Dynamic Few-Shot Assembly

```
ALGORITHM: DynamicFewShotAssembly

INPUT:
  q               : UserQuery
  D_pool          : ExamplePool (indexed in vector store with metadata)
  B_available     : AvailableTokenBudget (after role policy, retrieved evidence, instructions)
  k_max           : MaxExamples (default: 5)
  lambda_mmr      : MMRBalanceParameter (default: 0.7)
  include_negative: bool (default: true)

OUTPUT:
  D_selected      : OrderedList<Example>
  tokens_consumed : int

PROCEDURE:
  1. EMBED q using embedding model E
  2. RETRIEVE candidates ← VectorStore.search(emb(q), top_n = 3 * k_max, filters={domain, task_type})
  3. APPLY MMR selection:
       D_selected ← []
       WHILE |D_selected| < k_max AND B_available > 0:
         FOR each candidate c in candidates:
           score(c) = λ * sim(q, c.input) - (1-λ) * max_{s ∈ D_selected} sim(c.input, s.input)
         best ← argmax(score)
         IF tokens(best) ≤ B_available:
           APPEND best to D_selected
           B_available -= tokens(best)
           REMOVE best from candidates
         ELSE:
           BREAK
  4. IF include_negative AND B_available > tokens(smallest_negative_example):
       APPEND one contrastive negative example
       B_available -= tokens(negative_example)
  5. ORDER D_selected by increasing complexity (easy → hard)
  6. RETURN (D_selected, original_budget - B_available)
```

---

## 5. Combined CoT + Few-Shot: The Compound Technique

### 5.1 Architecture

The combination of CoT and Few-Shot is not additive — it is **multiplicative** in its effect on model performance. Few-shot examples that include explicit reasoning chains teach the model both **what** to produce and **how** to reason.

**Formal Representation:**

Each demonstration becomes a triple $(x_i, \mathcal{R}_i, y_i)$ where $\mathcal{R}_i$ is the reasoning chain. The model infers:

$$P(y \mid x, \mathcal{D}_{\text{CoT}}) = \sum_{\mathcal{R}} P(y \mid \mathcal{R}, x) \cdot P(\mathcal{R} \mid x, \{(x_i, \mathcal{R}_i, y_i)\}_{i=1}^k)$$

The demonstrations provide a strong prior over the structure, depth, and style of reasoning chains, while CoT ensures the model externalizes computation.

### 5.2 Production Template Structure

```
COMPILED PREFILL STRUCTURE:

[ROLE_POLICY]
  {role_description, behavioral_constraints, output_format_spec}

[COT_INSTRUCTION]
  {reasoning_mode: COMPRESSED, max_words_per_step: 5, operator_vocabulary: defined}

[FEW_SHOT_EXAMPLES]  ← dynamically assembled, ordered easy→hard
  EXAMPLE 1:
    Query: {x_1}
    Evidence: {E_1 with provenance tags}
    Reasoning: {R_1 in compressed format}
    Answer: {y_1}
  EXAMPLE 2:
    Query: {x_2}
    Evidence: {E_2 with provenance tags}
    Reasoning: {R_2 in compressed format}
    Answer: {y_2}
  [NEGATIVE EXAMPLE]:
    Query: {x_neg}
    Evidence: {E_neg}
    Reasoning: {R_neg — labeled as INCORRECT}
    Error: {explanation of failure mode}

[RETRIEVED_EVIDENCE]
  {current query's evidence, provenance-tagged, authority-ranked}

[CURRENT_QUERY]
  {q}

[OUTPUT_SCHEMA]
  {structured output specification: JSON schema or typed contract}
```

### 5.3 Token Budget Allocation Model

For a context window of $B_{\text{ctx}}$ tokens, the budget allocation follows a priority waterfall:

| Component | Priority | Typical Allocation |
|---|---|---|
| Role Policy + CoT Instruction | P0 (fixed) | 150–300 tokens |
| Output Schema | P0 (fixed) | 50–100 tokens |
| Retrieved Evidence (current query) | P1 (variable, high) | 30–50% of remaining |
| Few-Shot Examples | P2 (variable, medium) | 15–30% of remaining |
| Memory Summaries | P3 (variable, low) | 5–10% of remaining |
| Generation Budget | Reserved | 15–25% of $B_{\text{ctx}}$ |

$$B_{\text{evidence}} + B_{\text{examples}} + B_{\text{memory}} + B_{\text{fixed}} + B_{\text{gen}} \leq B_{\text{ctx}}$$

This allocation is computed at runtime by the **prefill compiler** and enforced as a hard invariant.

---

## 6. Tree of Thoughts (ToT): Formalization and SOTA Implementation

### 6.1 Theoretical Framework

Tree of Thoughts generalizes CoT from a **single linear chain** to a **search over a tree** of partial reasoning states. It treats LLM generation as a deliberate search problem.

**Formal Definition.** Let $\mathcal{S}$ be the space of intermediate thought states. A ToT process defines:

- **State generator** $G(s, q, \mathcal{E}) \rightarrow \{s_1, s_2, \ldots, s_b\}$: generates $b$ candidate next-states (branching factor)
- **State evaluator** $V(s, q, \mathcal{E}) \rightarrow \mathbb{R}$: scores each intermediate state for promise
- **Search algorithm** $\mathcal{A} \in \{\text{BFS}, \text{DFS}, \text{BeamSearch}, \text{MCTS}\}$: traverses the tree subject to budget

The solution is the leaf state with maximum accumulated value:

$$s^* = \arg\max_{s \in \text{Leaves}(\mathcal{T})} \sum_{s_i \in \text{path}(s)} V(s_i, q, \mathcal{E})$$

### 6.2 Search Strategies: Trade-off Analysis

| Strategy | Time Complexity | Space Complexity | Best Use Case |
|---|---|---|---|
| **BFS** (breadth-first) | $O(b^d)$ | $O(b^d)$ | Shallow trees, diverse exploration |
| **DFS** with pruning | $O(b \cdot d)$ | $O(d)$ | Deep reasoning, constrained budget |
| **Beam Search** ($w$) | $O(w \cdot b \cdot d)$ | $O(w \cdot d)$ | Balanced exploration/exploitation |
| **MCTS** (Monte Carlo Tree Search) | $O(n_{\text{simulations}})$ | $O(\text{tree size})$ | High-stakes, multi-evidence synthesis |

where $b$ = branching factor, $d$ = maximum depth, $w$ = beam width.

### 6.3 ToT for Multi-Document RAG Synthesis

In RAG systems with $n$ retrieved documents that may contain **conflicting, complementary, or redundant** information, ToT enables systematic evidence evaluation:

```
ALGORITHM: TreeOfThoughts_RAG

INPUT:
  q         : UserQuery
  E         : {e_1, ..., e_n} — provenance-tagged evidence documents
  b         : BranchingFactor (default: 3)
  d_max     : MaxDepth (default: 4)
  w         : BeamWidth (default: 2)
  B_llm     : LLMCallBudget (maximum number of LLM invocations)

OUTPUT:
  answer    : StructuredAnswer
  tree      : ReasoningTree (for audit)
  confidence: float

PROCEDURE:
  1. INITIALIZE root state s_0 = {q, E, partial_answer: ∅, reasoning: []}
  2. INITIALIZE beam = [s_0]
  3. FOR depth = 1 TO d_max:
       candidates ← []
       FOR each state s in beam:
         // GENERATE: produce b candidate next-thoughts
         thoughts ← LLM.generate_thoughts(s, b)
         // Each thought = one reasoning step (e.g., "Evidence e_3 contradicts e_1 on metric X")
         FOR each thought t in thoughts:
           s_new ← EXTEND(s, t)
           // EVALUATE: score the new state
           score ← LLM.evaluate_state(s_new, q, E)
           // Score considers: grounding, consistency, completeness, novelty
           candidates.append((s_new, score))
       // PRUNE: keep top-w candidates
       beam ← TopK(candidates, w, by=score)
       // EARLY TERMINATION: if top candidate is a complete answer with high confidence
       IF beam[0].is_complete AND beam[0].score > θ_complete:
         BREAK
       // BUDGET CHECK
       IF llm_calls_used >= B_llm:
         BREAK
  4. best_state ← beam[0]
  5. answer ← best_state.partial_answer
  6. tree ← RECONSTRUCT reasoning tree from all explored states
  7. confidence ← best_state.score
  8. RETURN (answer, tree, confidence)
```

### 6.4 Cost-Latency Analysis

ToT is **significantly more expensive** than linear CoT. The LLM call count scales as:

$$N_{\text{calls}} = \underbrace{w \cdot b \cdot d}_{\text{generation}} + \underbrace{w \cdot b \cdot d}_{\text{evaluation}} = 2 \cdot w \cdot b \cdot d$$

For $w=2, b=3, d=4$: $N_{\text{calls}} = 48$ LLM invocations per query.

**Mitigation Strategies:**

1. **Cached Evaluation:** Cache state evaluations by content hash; reuse across branches sharing prefixes.
2. **Smaller Evaluator Model:** Use a cheaper, faster model (e.g., GPT-4o-mini, Haiku) for state evaluation while using the full model for generation.
3. **Adaptive Branching:** Reduce $b$ dynamically when state evaluator confidence is high.
4. **Budget Ceiling:** Hard-cap $N_{\text{calls}}$ and return best-so-far with a degradation flag.

$$\text{Cost}_{\text{ToT}} \approx 2 \cdot w \cdot b \cdot d \cdot (\bar{T}_{\text{input}} \cdot C_{\text{in}} + \bar{T}_{\text{output}} \cdot C_{\text{out}})$$

where $\bar{T}$ are average token counts and $C$ are per-token costs.

**Decision Rule:** Deploy ToT only when $U_{\text{ToT}} - U_{\text{CoT}} > \text{Cost}_{\text{ToT}} - \text{Cost}_{\text{CoT}}$ in utility-adjusted terms. Reserve for high-stakes queries (financial analysis, medical reasoning, legal interpretation).

---

## 7. ReAct Prompting: Formalization for Agentic Systems

### 7.1 Theoretical Foundation

ReAct (Yao et al., 2023) interleaves **reasoning traces** (thoughts) with **actions** (tool invocations) in a single generation trajectory. This bridges the gap between chain-of-thought reasoning and agentic tool use.

**Formal Trajectory.** A ReAct execution produces a sequence:

$$\tau = (t_1, a_1, o_1, t_2, a_2, o_2, \ldots, t_n, a_n, o_n, y)$$

where:
- $t_i$ = thought (reasoning trace at step $i$)
- $a_i$ = action (tool invocation with typed parameters)
- $o_i$ = observation (tool return value, injected into context)
- $y$ = final answer

At each step $i$, the model generates:

$$t_i \sim P_\theta(\cdot \mid \mathcal{P}, t_{<i}, a_{<i}, o_{<i}, q)$$
$$a_i \sim P_\theta(\cdot \mid \mathcal{P}, t_{\leq i}, a_{<i}, o_{<i}, q)$$

The observation $o_i$ is **not generated** — it is the deterministic return value of executing $a_i$ against the external tool:

$$o_i = \text{Tool}_{a_i.\text{name}}(a_i.\text{params})$$

This is the critical architectural property: **observations are grounded in reality, not hallucinated.**

### 7.2 ReAct vs. Pure CoT vs. Pure Act: Comparative Analysis

| Dimension | Pure CoT | Pure Act (tool-only) | ReAct |
|---|---|---|---|
| Reasoning visibility | Full | None | Full |
| Grounding in external data | None (closed-book) | Full | Full |
| Hallucination risk | High (fabricates facts) | Low (but wrong tool selection) | Low (reasoning guides tool use) |
| Adaptability | Low (linear) | Low (no reasoning to adjust) | High (reasoning adjusts based on observations) |
| Token cost | Medium | Low per step | High (thoughts + actions + observations) |
| Error diagnosis | Easy | Hard | Easy |

### 7.3 Production ReAct Architecture with Typed Tool Contracts

In a production agentic system, ReAct must integrate with the typed protocol stack (MCP for discovery, JSON-RPC at boundaries, gRPC internally):

```protobuf
// Tool invocation contract (gRPC internal)
message ToolInvocationRequest {
  string tool_name = 1;
  string tool_version = 2;
  google.protobuf.Struct parameters = 3;
  string caller_agent_id = 4;
  string trace_id = 5;
  google.protobuf.Duration deadline = 6;
  AuthorizationContext auth = 7;
}

message ToolInvocationResponse {
  oneof result {
    google.protobuf.Struct data = 1;
    ToolError error = 2;
  }
  string provenance_id = 3;
  int64 latency_ms = 4;
  bool is_cached = 5;
}

message ToolError {
  string error_class = 1;    // TRANSIENT, PERMANENT, RATE_LIMITED, AUTH_DENIED
  string message = 2;
  bool is_retryable = 3;
  int32 retry_after_ms = 4;
}
```

### 7.4 Pseudo-Algorithm: Bounded ReAct Loop with Failure Recovery

```
ALGORITHM: BoundedReActLoop

INPUT:
  q             : UserQuery
  tools         : ToolRegistry (MCP-discovered, schema-described)
  P_react       : ReActPromptTemplate
  max_steps     : int (default: 8)
  B_ctx         : ContextWindowBudget
  retry_budget  : int (default: 2 per tool call)
  approval_gate : HumanApprovalPolicy

OUTPUT:
  answer        : StructuredAnswer
  trajectory    : List<(Thought, Action, Observation)>
  status        : {COMPLETED, BUDGET_EXHAUSTED, FAILED, ESCALATED}

PROCEDURE:
  1. INITIALIZE trajectory ← [], step ← 0, context_used ← tokens(P_react) + tokens(q)
  2. WHILE step < max_steps:
       step += 1
       
       // GENERATE thought + action
       (thought, action) ← LLM.generate(
         P_react + trajectory_summary + q,
         stop_sequences=["Observation:", "ANSWER:"]
       )
       
       // CHECK if model decided to emit final answer
       IF action.type == "FINAL_ANSWER":
         answer ← action.content
         RETURN (answer, trajectory, COMPLETED)
       
       // VALIDATE action against tool schema
       validation ← ToolRegistry.validate(action.tool_name, action.params)
       IF validation.failed:
         observation ← f"ERROR: Invalid parameters — {validation.errors}"
         trajectory.append((thought, action, observation))
         CONTINUE
       
       // APPROVAL GATE for state-changing actions
       IF action.is_mutating AND approval_gate.requires_approval(action):
         approval ← HumanApproval.request(action, timeout=approval_gate.timeout)
         IF NOT approval.granted:
           observation ← "ACTION DENIED by approval gate. Reason: {approval.reason}"
           trajectory.append((thought, action, observation))
           CONTINUE
       
       // EXECUTE tool with retry
       observation ← NULL
       FOR attempt = 1 TO retry_budget:
         response ← ToolRegistry.invoke(action, deadline=action.timeout)
         IF response.is_success:
           observation ← response.data
           BREAK
         ELIF response.error.is_retryable:
           WAIT(response.error.retry_after_ms * (1 + jitter()))
         ELSE:
           observation ← f"TOOL ERROR (permanent): {response.error.message}"
           BREAK
       IF observation IS NULL:
         observation ← "TOOL ERROR: Exhausted retry budget"
       
       trajectory.append((thought, action, observation))
       
       // CONTEXT MANAGEMENT: compress old trajectory entries if approaching budget
       context_used += tokens(thought) + tokens(action) + tokens(observation)
       IF context_used > 0.85 * B_ctx:
         trajectory_summary ← COMPRESS(trajectory, keep_last=3)
         context_used ← RECOMPUTE(P_react + trajectory_summary + q)
  
  3. // BUDGET EXHAUSTED
  RETURN (best_partial_answer(trajectory), trajectory, BUDGET_EXHAUSTED)
```

### 7.5 Context Window Management in ReAct Loops

Each ReAct step appends thought + action + observation tokens to the context window. Without management, the window saturates within ~5-8 steps for complex queries.

**SOTA Sliding Window Strategy:**

$$\text{ActiveContext} = \mathcal{P}_{\text{fixed}} \oplus \text{Summary}(\tau_{1:i-w}) \oplus \tau_{i-w+1:i} \oplus q$$

where $w$ is the window size (number of recent full steps retained) and $\text{Summary}(\cdot)$ is a compressed representation of older steps.

**Compression Function:**

$$\text{Summary}(\tau_{1:j}) = \text{LLM}_{\text{fast}}\left(\text{"Summarize key findings and remaining questions from these steps: "} \oplus \tau_{1:j}\right)$$

This is executed once when the window threshold is crossed, using a fast/cheap model, and the summary replaces the raw steps.

---

## 8. Prompting for Tool Usage: Typed Descriptions and Selection Optimization

### 8.1 The Tool Description as a Compiled Specification

In agentic systems, the LLM's tool selection accuracy is a **direct function of tool description quality**. A tool description is not documentation — it is a **contract specification** that the model uses for dispatch decisions.

### 8.2 SOTA Tool Description Schema

```typescript
interface ToolDescription {
  // IDENTITY
  name: string;                        // Active verb form: "get_current_weather"
  version: string;                     // Semantic versioning: "2.1.0"
  
  // PURPOSE (the model reads this for selection)
  description: string;                 // Concise, action-oriented, with boundary conditions
  
  // INPUT CONTRACT
  parameters: {
    type: "object";
    properties: Record<string, {
      type: string;
      description: string;            // Precise format specification
      enum?: string[];                 // Constrained values where applicable
      format?: string;                 // e.g., "date-time", "uri", "email"
      example?: any;                   // Concrete example value
    }>;
    required: string[];
  };
  
  // OUTPUT CONTRACT
  returns: {
    type: string;
    description: string;
    schema: JSONSchema;                // Full schema of return object
    example?: any;
  };
  
  // BEHAVIORAL CONTRACT
  side_effects: "none" | "read_only" | "mutating";
  requires_approval: boolean;
  timeout_class: "fast" | "medium" | "slow";  // <1s, <10s, <60s
  rate_limit: { calls_per_minute: number };
  
  // BOUNDARY CONDITIONS
  limitations: string[];               // Explicit constraints
  error_modes: string[];               // Known failure scenarios
  
  // SELECTION GUIDANCE
  use_when: string;                    // Positive selection criteria
  do_not_use_when: string;             // Negative selection criteria (anti-patterns)
}
```

### 8.3 Tool Selection as a Classification Problem

Formally, tool selection is a function:

$$f_{\text{select}}: (q, \mathcal{T}, \mathcal{C}) \rightarrow (t^*, \text{params}^*)$$

where $\mathcal{T} = \{T_1, \ldots, T_m\}$ is the tool registry and $\mathcal{C}$ is the current execution context.

**SOTA Optimization: Two-Phase Selection**

Phase 1: **Pre-filter** tools based on metadata matching (keyword, category, required permissions) to reduce $|\mathcal{T}|$ from potentially hundreds to $\leq 10$ relevant candidates. This is done outside the LLM.

Phase 2: **LLM-based selection** from the filtered candidate set, with full descriptions loaded into context only for relevant tools.

$$\text{Context cost} = O(k) \text{ where } k = |\mathcal{T}_{\text{filtered}}| \ll |\mathcal{T}|$$

This is critical for systems with large tool registries (>50 tools), where loading all descriptions would consume 30-60% of the context window.

### 8.4 Few-Shot Tool Usage Examples: Canonical Format

```
TOOL USAGE EXAMPLES:

Example 1 — Direct API lookup:
  User: "What's the weather like in Paris?"
  Thought: User wants current weather for a specific city. This maps to get_current_weather.
  Action: get_current_weather(city="Paris", units="celsius")
  Observation: {"high": 22, "low": 14, "conditions": "partly_cloudy", "humidity": 65}
  Answer: "It's partly cloudy in Paris today, with a high of 22°C and a low of 14°C."

Example 2 — Tool selection with disambiguation:
  User: "Find me a restaurant near the Eiffel Tower"
  Thought: User wants restaurant recommendations by location. This is a spatial search,
           not a weather query. Use search_restaurants, not get_current_weather.
  Action: search_restaurants(location="Eiffel Tower, Paris", radius_km=1, sort_by="rating")
  Observation: [{"name": "Le Jules Verne", "rating": 4.5, ...}, ...]
  Answer: "Here are top-rated restaurants near the Eiffel Tower: ..."

Example 3 — No tool needed:
  User: "What is 2 + 2?"
  Thought: This is a simple arithmetic question. No external tool needed.
  Action: NONE
  Answer: "4"
```

### 8.5 Pseudo-Algorithm: Tool-Augmented Prompt Compilation

```
ALGORITHM: ToolAugmentedPrefillCompilation

INPUT:
  q           : UserQuery
  T_registry  : FullToolRegistry
  P_base      : BasePromptTemplate
  B_ctx       : ContextWindowBudget
  B_gen       : GenerationBudget

OUTPUT:
  prefill     : CompiledPrefill
  tool_count  : int

PROCEDURE:
  1. CLASSIFY query intent → intent_categories (can be multi-label)
  2. FILTER tools:
       T_relevant ← T_registry.filter(
         categories ∈ intent_categories,
         permissions ⊆ caller.permissions,
         status = ACTIVE
       )
  3. IF |T_relevant| > k_max_tools (default: 8):
       // Rank by historical selection frequency for similar queries
       T_relevant ← TopK(T_relevant, k_max_tools, by=relevance_score(q, T))
  4. COMPUTE token cost:
       tool_tokens ← SUM(tokens(T.description) for T in T_relevant)
       fixed_tokens ← tokens(P_base) + tokens(few_shot_tool_examples)
       available_for_evidence ← B_ctx - B_gen - fixed_tokens - tool_tokens
  5. IF available_for_evidence < B_min_evidence:
       // Compress tool descriptions to short form
       T_relevant ← [T.compressed_description for T in T_relevant]
       RECOMPUTE tool_tokens
  6. ASSEMBLE prefill:
       prefill ← CONCAT(
         P_base.role_policy,
         P_base.cot_instruction,
         "AVAILABLE TOOLS:", T_relevant.descriptions,
         "TOOL USAGE EXAMPLES:", few_shot_tool_examples,
         "TOOL SELECTION RULES:", selection_policy,
         retrieved_evidence (if applicable),
         q
       )
  7. VALIDATE tokens(prefill) + B_gen ≤ B_ctx
  8. RETURN (prefill, |T_relevant|)
```

---

## 9. Prompt Compilation: The Prefill as a Runtime Artifact

### 9.1 Architectural Position

In a production agentic system, prompts are not hand-written strings — they are **compiled artifacts** assembled at runtime by a deterministic compiler. This compiler operates analogously to a traditional compiler:

| Traditional Compiler | Prefill Compiler |
|---|---|
| Source code → IR → machine code | Task spec → component assembly → token sequence |
| Linker resolves symbols | Retriever resolves evidence references |
| Optimizer eliminates dead code | Pruner eliminates low-relevance context |
| Output fits target architecture (ISA) | Output fits target context window ($B_{\text{ctx}}$) |

### 9.2 Prefill Compiler Pipeline

```
ALGORITHM: PrefillCompiler

INPUT:
  task_spec     : TaskSpecification
  role_policy   : RolePolicy (versioned)
  tools         : ToolRegistry
  memory        : MemoryLayers {working, session, episodic, semantic, procedural}
  retriever     : RetrievalEngine
  B_ctx         : ContextWindowBudget
  B_gen         : GenerationBudget

OUTPUT:
  prefill       : TokenSequence
  manifest      : CompilationManifest (for observability)

PROCEDURE:
  // PHASE 1: Budget allocation
  B_available ← B_ctx - B_gen
  allocations ← BudgetAllocator.compute(task_spec.complexity, B_available)
    // Returns: {role: X, tools: Y, evidence: Z, memory: W, examples: V}

  // PHASE 2: Component resolution
  role_section     ← RolePolicy.render(task_spec.domain, allocations.role)
  cot_instruction  ← CoTSelector.select(task_spec.reasoning_depth)
  tool_section     ← ToolCompiler.compile(task_spec, tools, allocations.tools)
  evidence_section ← Retriever.retrieve_and_rank(task_spec.query, allocations.evidence)
  memory_section   ← MemoryManager.summarize(task_spec, allocations.memory)
  examples_section ← FewShotAssembler.assemble(task_spec, allocations.examples)
  output_schema    ← SchemaCompiler.compile(task_spec.output_type)

  // PHASE 3: Assembly with priority-based truncation
  components ← [
    (role_section,     P0, FIXED),
    (cot_instruction,  P0, FIXED),
    (output_schema,    P0, FIXED),
    (evidence_section, P1, TRUNCATABLE),
    (examples_section, P2, TRUNCATABLE),
    (tool_section,     P2, TRUNCATABLE),
    (memory_section,   P3, DROPPABLE),
  ]
  
  prefill ← ""
  tokens_used ← 0
  FOR (component, priority, policy) in components SORTED BY priority:
    IF tokens_used + tokens(component) ≤ B_available:
      prefill ← prefill ⊕ component
      tokens_used += tokens(component)
    ELIF policy == TRUNCATABLE:
      truncated ← TRUNCATE(component, B_available - tokens_used)
      prefill ← prefill ⊕ truncated
      tokens_used ← B_available
      BREAK
    ELIF policy == DROPPABLE:
      SKIP component
      LOG warning: "Dropped {component.name} due to budget constraint"
    ELIF policy == FIXED:
      RAISE BudgetExceededError("Fixed component {component.name} exceeds remaining budget")

  // PHASE 4: Validation
  ASSERT tokens(prefill) + B_gen ≤ B_ctx
  ASSERT prefill contains role_section    // Invariant: role policy always present
  ASSERT prefill contains output_schema   // Invariant: output contract always present

  // PHASE 5: Manifest generation (observability)
  manifest ← {
    total_tokens: tokens_used,
    budget_utilization: tokens_used / B_available,
    components_included: [...],
    components_dropped: [...],
    evidence_count: len(evidence_section.documents),
    example_count: len(examples_section.examples),
    tool_count: len(tool_section.tools),
    compilation_latency_ms: timer.elapsed(),
  }

  RETURN (prefill, manifest)
```

### 9.3 Compilation Manifest: Observability Contract

Every compiled prefill produces a manifest that feeds into the observability pipeline:

```json
{
  "trace_id": "abc-123",
  "compilation_version": "3.2.1",
  "model_target": "gpt-4o-2024-08-06",
  "context_window": 128000,
  "generation_budget": 16000,
  "budget_utilization": 0.73,
  "components": {
    "role_policy": {"tokens": 180, "version": "v2.4", "status": "included"},
    "cot_instruction": {"tokens": 85, "mode": "compressed", "status": "included"},
    "output_schema": {"tokens": 62, "status": "included"},
    "evidence": {"tokens": 42000, "documents": 8, "avg_authority": 0.87, "status": "included"},
    "few_shot": {"tokens": 3200, "examples": 3, "selection": "MMR", "status": "included"},
    "tools": {"tokens": 1800, "count": 5, "status": "included"},
    "memory": {"tokens": 0, "status": "dropped", "reason": "budget_exhausted"}
  },
  "warnings": ["Memory section dropped due to budget constraint"],
  "compilation_latency_ms": 47
}
```

---

## 10. Framework-Augmented Prompt Optimization: DSPy, LlamaPromptOps, Synalinks

### 10.1 Decision Framework: When to Use Programmatic Prompt Optimization

| Criterion | Manual Prompting | Framework-Assisted |
|---|---|---|
| Query distribution complexity | Low-medium, well-understood | High, heterogeneous, evolving |
| Number of prompt variants to maintain | < 10 | > 10 |
| Evaluation infrastructure | Manual review | Automated benchmarks |
| Optimization objective | Qualitative ("good enough") | Quantitative (measurable metrics) |
| Team size | Individual/small | Platform team |
| Cost sensitivity | Low | High (optimization reduces cost) |

### 10.2 DSPy: Prompts as Differentiable Programs

DSPy (Khattab et al., 2023) treats prompt engineering as **programming**, not writing. Prompts are defined as typed **modules** with signatures, and the framework **compiles** them into optimized prompt strings or fine-tuning datasets through automated evaluation.

**Key Architectural Contribution:**

$$\text{DSPy Module}(x) = \text{Predict}[\text{Signature}](x) \quad \text{where Signature} = \text{Input} \rightarrow \text{Output}$$

The compiler uses **teleprompters** (optimizers) to:
1. Generate candidate prompt variations
2. Evaluate against labeled or LLM-judged benchmarks
3. Select optimal few-shot examples, instruction phrasings, and CoT strategies

**Integration with Agentic Stack:**
- DSPy modules can serve as individual agent steps in a plan → act → verify loop
- Signatures enforce typed input/output contracts compatible with the protocol stack
- Compiled prompts are versioned artifacts stored in the prefill compiler's registry

### 10.3 When to Avoid Frameworks

Frameworks add **dependency surface area**, **abstraction leakage**, and **debugging indirection**. Avoid them when:
- The prompt is stable and well-understood
- The team has strong prompt engineering expertise
- The system requires fine-grained control over every token in the context window
- The framework's optimization loop introduces unacceptable latency in the development cycle

---

## 11. Evaluation Infrastructure for Prompting Techniques

### 11.1 Metrics Taxonomy

| Metric Category | Specific Metrics | Measurement Method |
|---|---|---|
| **Correctness** | Exact match, F1, BLEU, ROUGE-L, BERTScore | Automated against gold labels |
| **Faithfulness** | Grounding score (% of claims supported by evidence) | NLI-based verification or LLM-as-judge |
| **Hallucination rate** | % of generated claims not traceable to context | Provenance verification pipeline |
| **Tool accuracy** | Tool selection precision/recall, parameter correctness | Schema validation + execution verification |
| **Reasoning quality** | Step validity, logical consistency, completeness | LLM-as-judge with rubric |
| **Token efficiency** | Tokens per correct answer, cost per query | Instrumented runtime |
| **Latency** | TTFT, total generation time, tool call latency | Distributed tracing |
| **Robustness** | Performance variance across paraphrases, adversarial inputs | Perturbation testing |

### 11.2 Pseudo-Algorithm: Continuous Prompting Evaluation Pipeline

```
ALGORITHM: ContinuousPromptEvaluation

INPUT:
  prompt_versions   : List<VersionedPrompt>
  eval_suite        : List<(query, gold_answer, gold_reasoning, metadata)>
  metrics           : List<MetricFunction>
  quality_gates     : Dict<MetricName, Threshold>
  model_configs     : List<ModelConfig>

OUTPUT:
  report            : EvaluationReport
  gate_status       : PASS | FAIL

PROCEDURE:
  FOR EACH prompt_v in prompt_versions:
    FOR EACH model_config in model_configs:
      results ← []
      FOR EACH (query, gold, gold_reasoning, meta) in eval_suite:
        // Compile prefill using prompt_v
        prefill ← PrefillCompiler.compile(prompt_v, query, model_config)
        // Execute
        (answer, reasoning, trace) ← AgentLoop.execute(prefill, model_config)
        // Evaluate
        scores ← {}
        FOR EACH metric in metrics:
          scores[metric.name] ← metric.compute(answer, gold, reasoning, gold_reasoning, trace)
        results.append(scores)
      
      // Aggregate
      aggregated ← AGGREGATE(results)  // mean, p50, p95, std
      
      // Gate check
      FOR EACH (metric_name, threshold) in quality_gates:
        IF aggregated[metric_name].mean < threshold:
          gate_status ← FAIL
          LOG "GATE FAILURE: {metric_name} = {aggregated[metric_name].mean} < {threshold}"
  
  report ← FORMAT(all_results, comparisons_across_versions)
  RETURN (report, gate_status)
```

### 11.3 CI/CD Integration

Prompt versions are treated as **code artifacts** in the deployment pipeline:

```
PIPELINE: PromptCI

trigger: on_pull_request(paths=["prompts/**", "tool_descriptions/**"])

stages:
  1. LINT:
       - Validate prompt template syntax
       - Check token budget compliance for all target models
       - Verify all tool references resolve in registry
  
  2. UNIT_TEST:
       - Run against minimal eval suite (< 50 cases, < 2 min)
       - Gate: faithfulness > 0.90, tool_accuracy > 0.95
  
  3. INTEGRATION_TEST:
       - Run against full eval suite (500+ cases, parallelized)
       - Compare against baseline prompt version (regression detection)
       - Gate: no metric regresses > 2% from baseline
  
  4. COST_ANALYSIS:
       - Compute average tokens_in, tokens_out per query
       - Compare against budget ceiling
       - Gate: cost_per_query < $0.05 (configurable)
  
  5. DEPLOY (on merge to main):
       - Version and tag prompt artifact
       - Update prefill compiler registry
       - Enable canary rollout (10% traffic)
       - Monitor production metrics for 2 hours
       - Auto-promote or rollback based on production quality gates
```

---

## 12. Hallucination Control Across Prompting Techniques

### 12.1 Hallucination Taxonomy in Prompted Systems

| Type | Description | Primary Mitigation via Prompting |
|---|---|---|
| **Intrinsic** | Model generates content contradicting input context | CoT with explicit evidence citation requirements |
| **Extrinsic** | Model generates content not present in any source | Grounding instructions: "Only use information from the provided documents" |
| **Tool fabrication** | Model invents tool names or parameters | Strict tool schema validation + few-shot correct usage examples |
| **Confidence fabrication** | Model expresses certainty about uncertain claims | Calibration instructions: "If unsure, say 'I don't have sufficient evidence'" |
| **Citation fabrication** | Model invents references or source IDs | Provenance-tagged evidence with verifiable IDs |

### 12.2 Anti-Hallucination Prompt Engineering Pattern

```
GROUNDING POLICY (included in every compiled prefill):

1. Base every claim on specific evidence from the RETRIEVED DOCUMENTS section.
2. Cite evidence by provenance ID (e.g., [DOC-3, §2.1]).
3. If retrieved documents do not contain sufficient information to answer,
   state: "Insufficient evidence in provided documents to answer this question."
4. Do not infer facts not explicitly stated or logically derivable from evidence.
5. If documents conflict, identify the conflict explicitly and prefer the source
   with higher authority score.
6. Never fabricate tool names, parameter values, URLs, citations, or statistics.
```

This policy is a **P0 fixed component** in the prefill compiler — it is never truncated or dropped, regardless of budget pressure.

---

## 13. Summary: Decision Matrix for Technique Selection

| Technique | Token Cost | LLM Calls | Best For | Avoid When |
|---|---|---|---|---|
| **Zero-Shot CoT** | +10-20 tokens | 1 | Simple reasoning tasks | Multi-step, evidence-heavy tasks |
| **Compressed CoT** | +40-120 tokens | 1 | Budget-constrained production | Tasks requiring detailed audit trails |
| **Manual CoT** | +100-300 tokens | 1 | Domain-specific critical reasoning | Diverse query distributions |
| **Few-Shot (k=3)** | +300-1500 tokens | 1 | Format/style conformance | Severe token budget constraints |
| **CoT + Few-Shot** | +400-2000 tokens | 1 | Maximum single-call accuracy | Real-time, low-latency requirements |
| **Tree of Thoughts** | +1000-5000 tokens | 10-50+ | High-stakes multi-evidence synthesis | Cost-sensitive, latency-sensitive queries |
| **ReAct** | +500-3000 tokens | 2-10+ | Tool-augmented agentic workflows | Pure reasoning without external data needs |

**Selection Rule:**

$$\text{Technique}^* = \arg\max_{T} \frac{U(T) - U(\text{baseline})}{\text{Cost}(T) + \lambda \cdot \text{Latency}(T)}$$

where $\lambda$ is the latency penalty coefficient set by SLA requirements.

---

## 14. Conclusion

Prompting techniques in production agentic systems are not isolated linguistic tricks — they are **architectural components** within a compiled context pipeline. Each technique carries measurable costs (tokens, latency, LLM calls, complexity) and measurable benefits (accuracy, faithfulness, tool-use correctness, reasoning transparency). Principal-level system design demands that these trade-offs be quantified, gated through evaluation infrastructure, and enforced through CI/CD pipelines that treat prompt artifacts with the same rigor as production code.

The key architectural principles are:

1. **Prompts are compiled, not written.** The prefill compiler assembles role policy, CoT instructions, few-shot examples, tool descriptions, retrieved evidence, and memory summaries into a deterministic artifact under a hard token budget.
2. **Every technique is subject to marginal utility analysis.** No technique is deployed without demonstrating $\Delta U / \Delta T$ above a configured threshold on the evaluation suite.
3. **Hallucination control is a P0 invariant.** Grounding policies are fixed components that survive all budget-driven truncation.
4. **Feedback loops are mechanical, not anecdotal.** Failed traces, human corrections, and production regressions are converted into evaluation suite entries and enforced through quality gates in CI/CD.
5. **Tool descriptions are typed contracts.** Tool selection accuracy depends on description quality, and descriptions must include boundary conditions, limitations, and contrastive selection guidance.